In [6]:
import numpy as np

In [7]:
import joblib
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

rf_model1 = joblib.load("../models/stage1_best_model.joblib")
rf_model2 = joblib.load("../models/stage2_best_model.joblib")

n_features_1 = rf_model1.n_features_in_
n_features_2 = rf_model2.n_features_in_
initial_type_1 = [("float_input", FloatTensorType([None, n_features_1]))]
initial_type_2 = [("float_input", FloatTensorType([None, n_features_2]))]

onnx_model_1 = convert_sklearn(
    rf_model1,
    initial_types=initial_type_1,
    target_opset=12
)

onnx_model_2 = convert_sklearn(
    rf_model2,
    initial_types=initial_type_2,
    target_opset=12
)

with open("stage1_random_forest.onnx", "wb") as f:
    f.write(onnx_model_1.SerializeToString())

with open("stage2_random_forest.onnx", "wb") as f:
    f.write(onnx_model_2.SerializeToString())

print("Saved stage1_random_forest.onnx")
print("Saved stage2_random_forest.onnx")

Saved stage1_random_forest.onnx
Saved stage2_random_forest.onnx


In [8]:
import onnxruntime as rt

sess = rt.InferenceSession("stage1_random_forest.onnx")

# Check inputs/outputs
print("Input:", sess.get_inputs()[0].name, sess.get_inputs()[0].shape)
print("Outputs:", [o.name for o in sess.get_outputs()])

# Run inference
X_test = np.random.rand(3, n_features_1).astype(np.float32)
labels, probas = sess.run(None, {"float_input": X_test})
print("Predicted labels:", labels)
print("Probabilities:", probas)

Input: float_input [None, 37]
Outputs: ['output_label', 'output_probability']
Predicted labels: [1 1 1]
Probabilities: [{0: -0.41384199261665344, 1: 0.41384199261665344}, {0: -0.41384199261665344, 1: 0.41384199261665344}, {0: -0.41384199261665344, 1: 0.41384199261665344}]
